# Partie 2 - Classification automatique des utilisateurs

On part du tableau de variables construit en partie 1 (une ligne par session, 616 variables numériques) et on cherche le modèle qui reconnaît le mieux l'utilisateur d'une session parmi 247.

**Plan :** protocole d'évaluation → comparaison des modèles → réglage des hyperparamètres → choix parmi les forêts → représentation « texte » des sessions (TF-IDF + SVM) → combinaison → améliorations (temps, augmentation, moyenne de SVM, règle du navigateur, combinaison géométrique) → analyse des erreurs → interprétation → prédiction sur le test et fichier de soumission.

## 1. Imports et chargement des données préparées

In [ ]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (AdaBoostClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid')

In [ ]:
# Racine du projet (le notebook est dans travail_partie_2/) ; données préparées dans donnees_preparees/
DATA_DIR = Path.cwd().parent
DOSSIER_PREPARE = DATA_DIR / 'donnees_preparees'

train = pd.read_csv(DOSSIER_PREPARE / 'train_variables.csv.gz', index_col='session')
X_test = pd.read_csv(DOSSIER_PREPARE / 'test_variables.csv.gz', index_col='session')

y = train.pop('util')
X = train

print(f'X : {X.shape[0]} sessions x {X.shape[1]} variables, {y.nunique()} utilisateurs')
print(f'X_test : {X_test.shape[0]} sessions x {X_test.shape[1]} variables')
assert list(X.columns) == list(X_test.columns)

## 2. Protocole d'évaluation

**Validation croisée stratifiée à 4 plis** (`StratifiedKFold`) : les sessions sont découpées en 4 parts ; chaque modèle est entraîné 4 fois sur 3 parts et évalué sur la part restante. Chaque session sert donc une fois en évaluation. *Stratifiée* : chaque part contient la même proportion de sessions de chaque utilisateur. C'est possible car chaque utilisateur a au moins 4 sessions.

**Deux métriques :**
- **accuracy** : part des sessions dont l'utilisateur est correctement prédit ;
- **F1 macro** : F1-score calculé pour chaque utilisateur, puis moyenné **sans pondération**. Un utilisateur à 4 sessions compte autant que `skm` et ses 75 sessions : c'est la bonne mesure quand les classes sont déséquilibrées.

**Référence** : un modèle « idiot » qui prédit toujours l'utilisateur le plus fréquent. Tout modèle utile doit faire nettement mieux.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
METRIQUES = ['accuracy', 'f1_macro']


def evaluer(modele):
    """Validation croisée : moyenne et écart-type des métriques sur les 4 plis, et temps de calcul."""
    debut = time.time()
    res = cross_validate(modele, X, y, cv=cv, scoring=METRIQUES)
    return {
        'accuracy': res['test_accuracy'].mean(),
        'accuracy_std': res['test_accuracy'].std(),
        'f1_macro': res['test_f1_macro'].mean(),
        'f1_macro_std': res['test_f1_macro'].std(),
        'temps_s': time.time() - debut,
    }

## 3. Comparaison des modèles

Les modèles proposés dans les indications, avec des réglages par défaut raisonnables. Les modèles sensibles à l'échelle des variables (régression logistique, SVM, kNN, MLP) sont précédés d'une **standardisation** (`StandardScaler` : moyenne 0, écart-type 1), placée dans un `pipeline` pour qu'elle soit apprise sur les plis d'entraînement uniquement. Les arbres n'en ont pas besoin.

*Durée : environ 5 minutes.*

In [ ]:
modeles = {
    'Référence (classe majoritaire)': DummyClassifier(strategy='most_frequent'),
    'Arbre de décision': DecisionTreeClassifier(random_state=0),
    'Naive Bayes (gaussien)': GaussianNB(),
    'Analyse discriminante linéaire (LDA)': LinearDiscriminantAnalysis(),
    'kNN (k = 1)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
    'SVM noyau linéaire': make_pipeline(StandardScaler(), SVC(kernel='linear', C=1)),
    'SVM noyau polynomial (deg. 2)': make_pipeline(StandardScaler(), SVC(kernel='poly', degree=2, C=10)),
    'SVM noyau RBF': make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10)),
    'Régression logistique': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'MLP (1 couche de 256)': make_pipeline(StandardScaler(), MLPClassifier((256,), max_iter=500, random_state=0)),
    'Random Forest (500 arbres)': RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=0),
    'Extra Trees (500 arbres)': ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=0),
}

resultats = {}
for nom, modele in modeles.items():
    resultats[nom] = evaluer(modele)
    print(f"{nom:32s} accuracy {resultats[nom]['accuracy']:.3f} | F1 macro {resultats[nom]['f1_macro']:.3f} "
          f"| {resultats[nom]['temps_s']:.0f} s")

resultats = pd.DataFrame(resultats).T.sort_values('f1_macro', ascending=False)
resultats.round(3)

In [ ]:
a_tracer = resultats.drop(index='Référence (classe majoritaire)')[['accuracy', 'f1_macro']].sort_values('f1_macro')

ax = a_tracer.plot.barh(figsize=(10, 6), color=['#2878b5', '#e07a5f'], xlim=(0, 1))
ax.set_title('Comparaison des modèles (validation croisée à 4 plis)')
ax.set_xlabel('Score')
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

**À retenir :**
- La référence (toujours prédire l'utilisateur le plus fréquent) n'obtient que **2,3 %** : le problème est difficile (247 classes).
- Les **forêts d'arbres** dominent nettement : **Extra Trees 93,3 %** et **Random Forest 92,5 %** d'accuracy, F1 macro ≈ 0,91.
- Le **Naive Bayes gaussien** est étonnamment bon (**83,6 %**, F1 0,80), devant la régression logistique, malgré son hypothèse naïve d'indépendance entre variables : il modélise le profil moyen de chaque utilisateur, ce qui correspond bien à nos proportions.
- Les modèles linéaires (régression logistique, SVM linéaire) et le MLP plafonnent autour de **80 %** ; la LDA fait moins bien (**71,9 %**), son hypothèse d'une même covariance pour tous les utilisateurs étant trop forte.
- Les SVM à noyau RBF ou polynomial et le kNN font moins bien : avec 616 variables dont beaucoup de proportions presque toujours nulles (écrans, configurations rares), la notion de distance entre sessions est peu pertinente.
- **Pourquoi les forêts ?** Nos variables agissent souvent comme des **seuils** (« utilise-t-il cet écran ? », « plus de 10 % de raccourcis ? ») et des **combinaisons** (navigateur + chaîne + type d'action) : c'est exactement ce que découpe un arbre. Moyenner des centaines d'arbres corrige l'instabilité d'un arbre seul (72,8 %).
- Les écarts-types entre plis sont faibles (< 1,5 point) : les classements sont fiables.

### 3.2 Les modèles de boosting

Le **boosting** construit les arbres **les uns après les autres**, chacun corrigeant les erreurs des précédents (alors qu'une forêt aléatoire les construit indépendamment et en fait la moyenne). On teste trois variantes :
- **AdaBoost** : repondère les sessions mal classées à chaque étape ;
- **XGBoost** : gradient boosting optimisé, souvent le meilleur sur des données tabulaires ;
- **HistGradientBoosting** : l'équivalent intégré à scikit-learn.

**Problème de coût :** avec 247 classes, XGBoost et HistGradientBoosting construisent **un arbre par classe et par itération**, soit 247 × 300 ≈ 74 000 arbres, en séquence. Une validation croisée complète prendrait plus de 30 minutes. On les évalue donc sur **un seul pli** (le premier de la validation croisée : 3/4 des sessions pour l'entraînement, 1/4 pour l'évaluation), avec la Random Forest retenue comme point de comparaison sur le même pli.

*Durée : environ 15 minutes.*

In [ ]:
# XGBoost demande des classes codées 0, 1, 2... : on encode les utilisateurs
encodeur = LabelEncoder()
y_code = encodeur.fit_transform(y)
idx_train, idx_eval = next(cv.split(X, y))  # premier pli de la validation croisée

boosting = {
    'Random Forest balanced (référence)': RandomForestClassifier(n_estimators=500, class_weight='balanced', n_jobs=-1, random_state=0),
    'AdaBoost (200 arbres de profondeur 10)': AdaBoostClassifier(DecisionTreeClassifier(max_depth=10, random_state=0),
                                                                  n_estimators=200, random_state=0),
    'XGBoost (300 itérations, profondeur 4)': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
                                                            colsample_bytree=0.3, tree_method='hist', n_jobs=-1, random_state=0),
    'HistGradientBoosting (200 itérations)': HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0),
}

resultats_boosting = {}
for nom, modele in boosting.items():
    debut = time.time()
    modele.fit(X.iloc[idx_train], y_code[idx_train])
    pred = modele.predict(X.iloc[idx_eval])
    resultats_boosting[nom] = {
        'accuracy': accuracy_score(y_code[idx_eval], pred),
        'f1_macro': f1_score(y_code[idx_eval], pred, average='macro'),
        'temps_s': time.time() - debut,
    }
    print(f"{nom:40s} accuracy {resultats_boosting[nom]['accuracy']:.3f} | "
          f"F1 macro {resultats_boosting[nom]['f1_macro']:.3f} | {resultats_boosting[nom]['temps_s']:.0f} s")

pd.DataFrame(resultats_boosting).T.round(3)

**À retenir (premier pli) :**
- **XGBoost** est le meilleur des boostings (**88,8 %**, F1 0,85) mais reste **6 points sous la Random Forest** sur le même pli (94,7 %, F1 0,94), pour un temps de calcul bien plus long (≈ 5 min contre ≈ 10 s).
- **AdaBoost** fait moins bien (67,6 %) : il se concentre sur les sessions difficiles, ce qui le rend sensible aux sessions atypiques, et il est mal adapté à 247 classes.
- **HistGradientBoosting** s'effondre (21 %) avec ces réglages : avec 247 classes et seulement 3 à 56 sessions d'entraînement par utilisateur, chaque arbre par classe dispose de très peu d'exemples positifs et l'apprentissage diverge. Il faudrait un taux d'apprentissage plus faible et de l'arrêt précoce, pour un coût encore plus élevé.
- **Conclusion :** sur ce problème (beaucoup de classes, peu d'exemples par classe), la **moyenne d'arbres indépendants** (bagging : Random Forest, Extra Trees) est plus efficace et plus robuste que le **boosting séquentiel**. On garde les forêts pour la suite.

## 4. Réglage des hyperparamètres

Les forêts (Random Forest, Extra Trees) et XGBoost sont les familles les plus prometteuses. On règle leurs hyperparamètres principaux :

| Hyperparamètre | Rôle |
|---|---|
| `class_weight` | `'balanced'` pondère chaque session à l'inverse de la fréquence de son utilisateur, pour que les utilisateurs ayant peu de sessions comptent autant que les autres |
| `max_features` | nombre de variables que chaque nœud peut considérer (`'sqrt'` ≈ 25 sur 616, ou une fraction). Moins il y en a, plus les arbres sont différents les uns des autres |
| `min_samples_leaf` | nombre minimal de sessions par feuille : plus il est grand, plus les arbres sont simples (moins de sur-apprentissage) |
| `n_estimators` | nombre d'arbres |
| `max_depth`, `learning_rate` (XGBoost) | profondeur de chaque arbre, et part de correction apportée par chaque nouvel arbre |

La recherche prend environ une heure : elle est faite par le script `recherche_hyperparametres.py` (dans ce dossier), qui enregistre ses résultats dans `resultats/`. On relit ici ces résultats. Les forêts sont évaluées par **validation croisée à 4 plis** sur une grille complète ; XGBoost, trop coûteux avec 247 classes, sur le **premier pli** seulement.

In [ ]:
DOSSIER_RESULTATS = Path.cwd() / 'resultats'
grille_rf = pd.read_csv(DOSSIER_RESULTATS / 'recherche_random_forest.csv')
grille_et = pd.read_csv(DOSSIER_RESULTATS / 'recherche_extra_trees.csv')
grille_arbres = pd.read_csv(DOSSIER_RESULTATS / 'recherche_nombre_arbres.csv')
grille_xgb = pd.read_csv(DOSSIER_RESULTATS / 'recherche_xgboost_pli1.csv')

for grille in (grille_rf, grille_et, grille_arbres):
    if 'param_class_weight' in grille:
        grille['param_class_weight'] = grille['param_class_weight'].fillna('aucune')
    grille.columns = grille.columns.str.replace('param_', '').str.replace('mean_test_', '')

print('Random Forest : 5 meilleures combinaisons (sur 18)')
display(grille_rf.head(5).round(3))
print('Extra Trees : 5 meilleures combinaisons (sur 16)')
display(grille_et.head(5).round(3))

In [ ]:
# Effet moyen de chaque hyperparamètre sur le F1 macro
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
for ligne, (nom, grille) in enumerate([('Random Forest', grille_rf), ('Extra Trees', grille_et)]):
    for col, param in enumerate(['class_weight', 'max_features', 'min_samples_leaf']):
        moyenne = grille.groupby(param)['f1_macro'].mean()
        moyenne.plot.bar(ax=axes[ligne, col], color='#2878b5' if ligne == 0 else '#e07a5f', rot=0)
        axes[ligne, col].set_ylim(0.88, 0.95)
        axes[ligne, col].set_title(f'{nom} : {param}')
        axes[ligne, col].set_xlabel('')
        for conteneur in axes[ligne, col].containers:
            axes[ligne, col].bar_label(conteneur, fmt='%.3f', fontsize=8)
axes[0, 0].set_ylabel('F1 macro moyen')
axes[1, 0].set_ylabel('F1 macro moyen')
plt.tight_layout()
plt.show()

In [ ]:
print('Random Forest balanced : effet du nombre d arbres')
display(grille_arbres.sort_values('n_estimators')[['n_estimators', 'accuracy', 'f1_macro', 'mean_fit_time']].round(3))
print('XGBoost (premier pli) : la Random Forest retenue obtient 0,947 / 0,940 sur ce même pli')
display(grille_xgb.round(3))

**À retenir :**
- **`class_weight='balanced'` est le réglage décisif** : en moyenne sur les grilles, le F1 macro passe de 0,909 à **0,940** pour la Random Forest et de 0,914 à **0,934** pour les Extra Trees. Sans pondération, les forêts favorisent les utilisateurs qui ont beaucoup de sessions. La variante `balanced_subsample` (pondération recalculée pour chaque arbre) fait moins bien (0,926).
- **`max_features`** a peu d'effet sur la Random Forest (0,923 à 0,927), mais les Extra Trees ont besoin de plus de variables par nœud (0,910 avec `sqrt`, **0,934** avec 0,15) : comme leurs seuils sont tirés au hasard, ils compensent en examinant plus de variables.
- **`min_samples_leaf`** n'a pratiquement pas d'effet (± 0,4 point).
- **Nombre d'arbres** : le score monte jusqu'à **500 arbres** puis se stabilise (1 000 n'apporte rien et double le temps de calcul).
- **XGBoost** plafonne à **0,880** d'accuracy (F1 0,847) sur le premier pli, contre 0,947 pour la Random Forest, et il est environ 30 fois plus lent. Un taux d'apprentissage plus fort (0,3) le dégrade, une profondeur plus grande n'apporte rien, la pondération non plus. **On l'écarte.**

## 5. Choix parmi les forêts

On compare, en validation croisée, le meilleur réglage de chaque forêt et leur **combinaison** : un `VotingClassifier` en vote « souple » (`voting='soft'`), qui fait la **moyenne des probabilités** des deux forêts pour chaque utilisateur. La Random Forest (seuils optimisés) et les Extra Trees (seuils tirés au hasard) ne font pas tout à fait les mêmes erreurs : la moyenne peut en corriger une partie.

*Durée : environ 4 minutes.*

In [ ]:
meilleure_rf = RandomForestClassifier(n_estimators=500, class_weight='balanced', max_features='sqrt',
                                      min_samples_leaf=1, n_jobs=-1, random_state=0)
meilleurs_et = ExtraTreesClassifier(n_estimators=500, class_weight='balanced', max_features=0.15,
                                    min_samples_leaf=2, n_jobs=-1, random_state=0)
ensemble = VotingClassifier([('rf', meilleure_rf), ('et', meilleurs_et)], voting='soft')

choix = {}
for nom, modele in [('Random Forest (meilleur réglage)', meilleure_rf),
                    ('Extra Trees (meilleur réglage)', meilleurs_et),
                    ('Ensemble RF + ET (vote souple)', ensemble)]:
    choix[nom] = evaluer(modele)
    print(f"{nom:34s} accuracy {choix[nom]['accuracy']:.3f} | F1 macro {choix[nom]['f1_macro']:.3f}")

pd.DataFrame(choix).T.round(3)

**À retenir :**
- Les trois candidats sont très proches, mais l'ordre est le même sur les deux métriques : Random Forest (0,946 / 0,942) < Extra Trees (0,949 / 0,945) < **ensemble (0,951 / 0,947)**, avec des écarts-types faibles (≤ 0,5 point).
- Les soumissions Kaggle ont confirmé cet ordre (0,920 pour la RF, 0,930 pour les ET, **0,938** pour l'ensemble) : la validation croisée est un guide fiable.
- **On retient l'ensemble RF + ET comme meilleur modèle sur les variables agrégées** : c'est la composante « forêts » du modèle final.

In [ ]:
forets = ensemble
classes = np.sort(y.unique())  # ordre des colonnes de probabilités, identique pour tous les modèles

## 6. Une seconde représentation : les sessions comme des textes

**Pourquoi ?** Plusieurs équipes atteignent 0,98 sur Kaggle : il reste du signal. Or nos 616 variables **résument** chaque session (proportions par famille) et laissent de côté :
- les **valeurs rares** (présentes dans une seule session) ;
- les **combinaisons exactes** d'une action : `Exécution d'un bouton<DEF_03/24>$JCP$` est découpé en un type, une configuration et une chaîne, et la combinaison est perdue ;
- l'**ordre** des actions (des paires de *types* d'actions n'apportaient rien, voir les notes du rapport, mais les paires d'actions *complètes* n'avaient pas été testées).

**Idée :** traiter chaque session comme un **document**, dont les « mots » sont :
- le navigateur ;
- chaque **action brute** (type + écran + configuration + chaîne + suffixe) ;
- chaque **paire d'actions consécutives** (`action_1 >> action_2`) ;
- les **composants séparés** de chaque action (`TYPE=…`, `ECRAN=…`, `CONF=…`, `CHAINE=…`), pour que deux actions sur le même écran se « ressemblent ».

On vectorise ces documents par **TF-IDF** : un mot reçoit un poids fort s'il est **fréquent dans la session** mais **rare dans l'ensemble des sessions** (les signatures), faible s'il est utilisé par tout le monde. `sublinear_tf=True` atténue les mots répétés des centaines de fois. On obtient des dizaines de milliers de colonnes très creuses, pour lesquelles un **SVM linéaire** (`LinearSVC`) est rapide et efficace.

Le code de cette représentation est dans `representation_texte.py` (dans ce dossier).

In [ ]:
from representation_texte import charger_documents, combiner, mots_session, svm_tfidf, trouver_fichier

documents_train = charger_documents(trouver_fichier(DATA_DIR, 'train'), X.index, avec_util=True)
documents_test = charger_documents(trouver_fichier(DATA_DIR, 'test'), X_test.index, avec_util=False)
print(f'{len(documents_train)} documents de train, {len(documents_test)} documents de test')

# Exemple : les premiers mots de chaque sorte pour la première session
mots = mots_session(documents_train[0])
print('Nombre de mots de la session 0 :', len(mots))
print('Navigateur et actions :', mots[:4])
print('Paires :', [m for m in mots if ' >> ' in m][:2])
print('Composants :', [m for m in mots if m.startswith(('ECRAN=', 'CHAINE='))][:3])

**Variantes testées** (script `recherche_tfidf_combinaison.py`, validation croisée à 4 plis, même découpage que le reste du notebook) :

In [ ]:
variantes = pd.read_csv(DOSSIER_RESULTATS / 'tfidf_variantes.csv')
variantes.round(3)

**À retenir :**
- Les **actions brutes seules** donnent 0,921 : moins bien que nos forêts (0,951).
- Ajouter les **paires d'actions consécutives** fait gagner **3,5 points** (0,956). L'ordre des actions compte donc, **à condition de garder tout le détail de l'action** : les paires de simples *types* d'actions n'apportaient rien.
- Les **composants séparés** aident encore un peu (0,957, F1 0,949) ; les **triplets** n'apportent rien (trop rares, ils diluent le signal) ; `C = 0,3` (régularisation plus forte) est moins bon.
- `min_df = 2` (ignorer les mots vus dans une seule session) est équivalent (0,958 / 0,950, dans le bruit). On garde **actions + paires + composants**, `C = 1`.
- Seul, le SVM a une meilleure accuracy que les forêts (0,957 contre 0,951) mais un F1 macro légèrement inférieur (0,949 contre 0,950) : il reconnaît mieux les sessions courantes, moins bien certains petits utilisateurs.

## 7. Combinaison du SVM TF-IDF et des forêts

Les deux modèles reposent sur des représentations très différentes : les forêts exploitent les **proportions et le rythme**, le SVM les **combinaisons exactes et l'ordre** des actions. Leurs erreurs devraient donc être en partie différentes, et leur combinaison meilleure que chacun.

**Méthode :**
- le SVM ne donne pas de probabilités, mais des scores de décision. On les transforme en pseudo-probabilités par un **softmax** : $p_k = e^{s_k / T} / \sum_j e^{s_j / T}$. La **température** $T$ règle à quel point elles sont tranchées ;
- on fait la **moyenne pondérée** : $p = w \cdot p_{SVM} + (1 - w) \cdot p_{forêts}$, puis on prédit l'utilisateur de plus forte probabilité.

$T$ et $w$ sont choisis par validation croisée **sur le train uniquement**. Comme les écarts attendus sont de l'ordre du bruit de la mesure, la validation croisée est **répétée sur 3 découpages différents** (graines 0, 1, 2), avec 6 × 6 combinaisons de $(T, w)$.

In [ ]:
combinaison = pd.read_csv(DOSSIER_RESULTATS / 'combinaison.csv')
combinaison['version'] = combinaison.apply(
    lambda r: f"combinaison T={r.temperature:g}, w={r.poids_svm:g}" if r.version == 'combinaison' else r.version, axis=1)
resume = (combinaison.groupby('version')[['accuracy', 'f1_macro']].agg(['mean', 'std'])
          .sort_values(('f1_macro', 'mean'), ascending=False).round(4))
print('Moyenne et écart-type sur les 3 découpages (10 meilleures versions + modèles seuls)')
display(pd.concat([resume.head(10), resume.loc[['forêts seules', 'SVM TF-IDF seul']]]))

grille = (combinaison[combinaison['temperature'].notna()]
          .groupby(['temperature', 'poids_svm'])['f1_macro'].mean().unstack())
plt.figure(figsize=(8, 5))
sns.heatmap(grille, annot=True, fmt='.3f', cmap='Blues')
plt.title('F1 macro moyen (3 découpages) selon la température et le poids du SVM')
plt.xlabel('Poids du SVM (w)')
plt.ylabel('Température (T)')
plt.tight_layout()
plt.show()

**À retenir :**
- **La combinaison bat nettement chaque modèle seul**, sur les 3 découpages : F1 macro **0,965** contre 0,951 (forêts) et 0,947 (SVM), soit **+1,4 point**, environ 5 fois l'écart-type entre découpages. L'erreur passe de 4,9 % à 3,2 %.
- La carte montre un **plateau** autour de T = 0,2 à 0,3 et w = 0,5 à 0,8. Une température trop forte (T = 1) rend les probabilités du SVM presque uniformes : il ne pèse plus rien et on retrouve le score des forêts.
- **Réglage retenu : T = 0,2, w = 0,6**, au centre du plateau et avec l'écart-type le plus faible, plutôt que le maximum isolé (T = 0,3, w = 0,8), qui pourrait être un effet du hasard.
- **Confirmé sur Kaggle : 0,969** (contre 0,938 pour les forêts seules).

In [ ]:
TEMPERATURE = 0.2
POIDS_SVM = 0.6
svm = svm_tfidf(paires=True, composants=True)
svm

## 8. Améliorations du modèle combiné

Après la combinaison de la section 7 (0,969 sur Kaggle), plusieurs pistes ont été testées. Comme les gains attendus sont de quelques dixièmes de point, chaque piste est évaluée sur **3 découpages** et n'est retenue que si elle **bat la référence sur les 3**. Les calculs (≈ 45 min) sont faits par le script `recherche_ameliorations.py`, qui enregistre `resultats/ameliorations.csv`.

**Pistes retenues :**
- **Règle du navigateur** : dans le train, chaque utilisateur utilise toujours le même navigateur. On met donc à zéro la probabilité des utilisateurs qui n'ont jamais utilisé le navigateur de la session. Le SVM, qui ne voit le navigateur que comme un mot parmi des milliers, pouvait sinon proposer un utilisateur incompatible.
- **Le temps dans le texte** : les repères `tXX` deviennent un mot `TEMPS`, gardé à sa place dans la séquence. Les paires « action >> TEMPS » indiquent les actions suivies d'un passage de 5 secondes : le **rythme** de l'utilisateur.
- **Un SVM moins régularisé** (C = 10) : avec près de 100 000 mots, il peut coller davantage aux détails sans sur-apprendre.
- **L'augmentation de données** : un utilisateur à 4 sessions n'a que 3 exemples d'entraînement par pli. On ajoute, pour chaque session d'entraînement, **8 extraits** de 30 à 200 éléments : autant de « sessions courtes » supplémentaires du même utilisateur. Le modèle apprend ainsi à reconnaître les sessions courtes, où se concentrent les erreurs. L'augmentation n'est appliquée **qu'aux plis d'entraînement** (jamais aux sessions évaluées).
- **La moyenne de plusieurs SVM** : trois SVM qui voient les sessions sous des angles différents (avec temps, avec temps + augmentation, sans temps) ; la moyenne de leurs probabilités est plus stable que chacun.

**Pistes écartées** (détail dans les notes du rapport) : TF-IDF binaire, SVM sans pondération des classes, régression logistique et plus proches voisins sur le TF-IDF, méta-modèle (*stacking*) pour combiner les modèles, mots de navigation entre écrans, mots de rythme, Naive Bayes.

**Après la v3**, trois pistes supplémentaires ont été retenues, toujours avec la règle des 3 découpages :
- **Plus de SVM dans la moyenne (v4)** : 3 SVM augmentés de plus (deux autres tirages des 8 extraits de 30 à 200 éléments, et 15 extraits de 20 à 150), et un SVM **sans IDF** (sans pondération des mots rares), soit **7 SVM**. Ils se trompent sur des sessions un peu différentes, et leur moyenne est plus stable.
- **Correction des fréquences** : on divise la probabilité de chaque utilisateur par sa fréquence dans le train à la puissance α = 0,1. Cela favorise légèrement les petits utilisateurs, ce qui va dans le sens du F1 macro, où chaque utilisateur compte autant.
- **Combinaison géométrique (v5)** : au lieu de la moyenne des probabilités, on fait la moyenne des **log-probabilités** : $score = 0{,}8 \cdot \overline{\log p_{SVM}} + 0{,}2 \cdot \log(p_{forêts} + 0{,}01) - 0{,}1 \cdot \log(\text{fréquence})$. Un utilisateur jugé très improbable par un des modèles est fortement pénalisé : il faut que **tous** les modèles soient d'accord. Cela permet de donner plus de poids aux forêts (20 % au lieu de 5 % pour la v4).

**Autres pistes écartées après la v3** : autres formulations du SVM linéaire (Ridge, Crammer-Singer), plus proches voisins et Extra Trees seuls ajoutés à l'ensemble, autres transformations des comptages de mots (racine carrée, BM25, norme L1), sélection des mots par χ², méta-modèles (*stacking*, choix du modèle session par session), sélection gloutonne d'un sous-ensemble de modèles, réseau de neurones (MLP, 0,909 seul).

In [ ]:
ameliorations = pd.read_csv(DOSSIER_RESULTATS / 'ameliorations.csv')
principales = ameliorations[~ameliorations['version'].str.startswith('grille')]
tableau = principales.pivot_table(index='version', columns='graine', values='f1_macro', sort=False)
tableau.columns = [f'F1 découpage {g}' for g in tableau.columns]
tableau['F1 moyen'] = tableau.mean(axis=1)
tableau['accuracy moyenne'] = principales.groupby('version', sort=False)['accuracy'].mean()
display(tableau.round(4))

grille_v3 = ameliorations[ameliorations['version'].str.startswith('grille')].copy()
grille_v3[['T', 'w']] = grille_v3['version'].str.extract(r'T=([\d.]+), w=([\d.]+)').astype(float)
carte = grille_v3.groupby(['T', 'w'])['f1_macro'].mean().unstack()
plt.figure(figsize=(7, 4))
sns.heatmap(carte, annot=True, fmt='.4f', cmap='Blues')
plt.title('v3 : F1 macro moyen (3 découpages) selon T et w')
plt.xlabel('Poids des SVM (w)')
plt.ylabel('Température (T)')
plt.tight_layout()
plt.show()

**À retenir :**
- **Règle du navigateur** : 0,965 → **0,967**, gratuitement.
- **v2** (SVM avec temps, C = 10, + forêts + navigateur) : **0,971** de F1 macro, gain sur les 3 découpages. **0,980 sur Kaggle.**
- Le SVM **augmenté** est le meilleur SVM seul (0,952), mais combiné seul aux forêts son gain varie trop selon le découpage (+0,59 / +0,13 / −0,14 point) : c'est en **moyenne avec les deux autres SVM** qu'il est utile.
- **v3** (moyenne de 3 SVM + forêts + navigateur) : **0,978 d'accuracy, F1 macro 0,975**, gain sur les 3 découpages. **0,985 sur Kaggle.**
- La carte montre un plateau pour w = 0,8 à 0,9 : avec trois SVM moyennés, leurs probabilités sont plus fiables et méritent plus de poids que les forêts. Réglage retenu : **T = 0,5, w = 0,9**.
- **v4** (7 SVM + forêts, T = 0,7, w = 0,95, navigateur, α = 0,1) : **0,981 d'accuracy, F1 macro 0,979** (3 découpages). **0,985 sur Kaggle**, comme la v3.
- **v5** (les mêmes modèles, combinaison géométrique) : **F1 macro 0,980**, gain sur la v4 de +0,15 / +0,08 / +0,06 point selon le découpage. Elle ne change que 3 prédictions du test, mais obtient **0,991 sur Kaggle** : c'est le **modèle final**.
- Au-delà de 0,97, chaque piste rapporte quelques dixièmes de point et le premier découpage est souvent trompeur : sans la validation répétée, on aurait retenu de fausses améliorations.

In [ ]:
from representation_texte import augmenter, combiner_geometrique, masque_navigateur, softmax

# Documents avec le mot TEMPS (les documents sans temps de la section 6 restent utilisés par le SVM C = 3)
documents_temps_train = charger_documents(trouver_fichier(DATA_DIR, 'train'), X.index, avec_util=True, avec_temps=True)
documents_temps_test = charger_documents(trouver_fichier(DATA_DIR, 'test'), X_test.index, avec_util=False, avec_temps=True)

# v3 : 3 SVM + forêts + règle du navigateur (moyenne des probabilités)
T_V3, W_V3 = 0.5, 0.9
OPTIONS_AUGMENTATION = dict(n=8, long_min=30, long_max=200)

# Modèle final (v5) : 7 SVM + forêts, combinaison géométrique, règle du navigateur, correction des fréquences
T_FINAL, POIDS_FORETS_FINAL, ALPHA_FINAL = 1.0, 0.2, 0.1
# SVM augmentés : (nom, décalage de la graine du tirage des extraits, options des extraits)
AUGMENTATIONS = [('svm_temps_C10_augmente', 0, OPTIONS_AUGMENTATION),
                 ('svm_temps_C10_augmente_r1', 1000, OPTIONS_AUGMENTATION),
                 ('svm_temps_C10_augmente_r2', 2000, OPTIONS_AUGMENTATION),
                 ('svm_temps_C10_augmente_15x20_150', 0, dict(n=15, long_min=20, long_max=150))]
log_frequence = np.log(y.value_counts().reindex(classes).values / len(y))


def score_final(scores_svm, proba_forets, masque):
    """Score de la v5 (à maximiser) : combinaison géométrique, correction des fréquences, règle du navigateur."""
    score = combiner_geometrique(scores_svm, proba_forets, T_FINAL, POIDS_FORETS_FINAL) - ALPHA_FINAL * log_frequence
    return np.where(masque > 0, score, -np.inf)


nav_cols = [c for c in X.columns if c.startswith('nav_')]
navigateur_train = X[nav_cols].idxmax(axis=1)
navigateur_par_utilisateur = navigateur_train.groupby(y).first()
masque_train = masque_navigateur(navigateur_train, navigateur_par_utilisateur, classes)
masque_test = masque_navigateur(X_test[nav_cols].idxmax(axis=1), navigateur_par_utilisateur, classes)
print('Exemple de document avec temps :', documents_temps_train[1][:8])

## 9. Analyse des erreurs du modèle final

Prédictions **hors échantillon** sur le découpage de la section 2 : chaque session est prédite par des modèles qui ne l'ont pas vue. Pour le SVM augmenté, la validation croisée est faite à la main, pour n'augmenter que les plis d'entraînement. Si `recherche_ameliorations.py` a déjà été lancé, ses prédictions (même découpage, mêmes modèles) sont relues dans `resultats/cache_oof/` ; sinon elles sont calculées ici.

*Durée : environ 10 minutes sans le cache.*

In [ ]:
CACHE_OOF = DOSSIER_RESULTATS / 'cache_oof'


def predictions_hors_echantillon(nom, calcul):
    fichier = CACHE_OOF / f'{nom}_g0.npy'
    if fichier.exists():
        return np.load(fichier)
    resultat = calcul()
    CACHE_OOF.mkdir(parents=True, exist_ok=True)
    np.save(fichier, resultat)
    return resultat


def svm_augmente_hors_echantillon(graine, options):
    rng = np.random.default_rng(graine)
    scores = np.zeros((len(y), len(classes)))
    y_liste = list(y)
    for idx_train, idx_eval in cv.split(X, y):
        docs, ys = augmenter([documents_temps_train[i] for i in idx_train], [y_liste[i] for i in idx_train], rng,
                             **options)
        modele = svm_tfidf(C=10).fit(docs, ys)
        scores[idx_eval] = modele.decision_function([documents_temps_train[i] for i in idx_eval])
    return scores


proba_forets_cv = predictions_hors_echantillon(
    'forets', lambda: cross_val_predict(forets, X, y, cv=cv, method='predict_proba'))
scores_svm_cv = predictions_hors_echantillon(
    'svm', lambda: cross_val_predict(svm, documents_train, y, cv=cv, method='decision_function'))
scores_temps_cv = predictions_hors_echantillon(
    'svm_temps_C10', lambda: cross_val_predict(svm_tfidf(C=10), documents_temps_train, y, cv=cv, method='decision_function'))
scores_augmentes_cv = [predictions_hors_echantillon(nom, lambda g=graine, o=options: svm_augmente_hors_echantillon(g, o))
                       for nom, graine, options in AUGMENTATIONS]
scores_c3_cv = predictions_hors_echantillon(
    'svm_C3', lambda: cross_val_predict(svm_tfidf(C=3), documents_train, y, cv=cv, method='decision_function'))
scores_sans_idf_cv = predictions_hors_echantillon(
    'svm_temps_C10_sans_idf',
    lambda: cross_val_predict(svm_tfidf(C=10, use_idf=False), documents_temps_train, y, cv=cv, method='decision_function'))
scores_v5_cv = [scores_temps_cv, *scores_augmentes_cv, scores_c3_cv, scores_sans_idf_cv]

predictions_cv = {
    'Forêts seules': proba_forets_cv,
    'SVM + forêts (section 7)': combiner(scores_svm_cv, proba_forets_cv, TEMPERATURE, POIDS_SVM),
    'v3 (3 SVM)': combiner([scores_temps_cv, scores_augmentes_cv[0], scores_c3_cv], proba_forets_cv,
                           T_V3, W_V3) * masque_train,
    'Modèle final (v5)': score_final(scores_v5_cv, proba_forets_cv, masque_train),
}
predictions_cv = {nom: classes[p.argmax(axis=1)] for nom, p in predictions_cv.items()}
for nom, pred in predictions_cv.items():
    print(f'{nom:26s} accuracy {accuracy_score(y, pred):.3f} | F1 macro {f1_score(y, pred, average="macro"):.3f} '
          f'| erreurs {(pred != y).sum()}')
y_pred_cv = predictions_cv['Modèle final (v5)']

In [ ]:
# Où se font les gains ? Accuracy selon la longueur de la session
nb_actions = np.expm1(X['log_nb_actions']).round()
tranches = pd.cut(nb_actions, [0, 50, 100, 200, 500, 1000, np.inf],
                  labels=['≤ 50', '51-100', '101-200', '201-500', '501-1000', '> 1000'])
par_longueur = pd.DataFrame({'sessions': tranches.value_counts().sort_index()})
for nom, pred in predictions_cv.items():
    par_longueur[nom] = pd.Series(pred == y.values, index=y.index).groupby(tranches).mean()
display(par_longueur.round(3))

ax = par_longueur[list(predictions_cv)].plot.bar(figsize=(10, 4), rot=0, color=['#2878b5', '#e07a5f', '#f2cc8f', '#4d908e'])
ax.set_ylim(0.6, 1)
ax.set_title('Accuracy selon le nombre d actions de la session')
ax.set_xlabel('Nombre d actions')
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
rapport = pd.DataFrame(classification_report(y, y_pred_cv, output_dict=True, zero_division=0)).T
rapport = rapport.drop(index=['accuracy', 'macro avg', 'weighted avg'])
rapport['support'] = rapport['support'].astype(int)

print('Les 10 utilisateurs les moins bien reconnus')
display(rapport.sort_values('f1-score').head(10).round(2))

erreurs = pd.DataFrame({'vrai': y, 'predit': y_pred_cv})[y != y_pred_cv]
print('Les confusions les plus fréquentes (vrai -> prédit)')
display(erreurs.value_counts().head(10).rename('nb de sessions').to_frame())

**À retenir :**
- **Modèle final (v5) : 98,1 % d'accuracy, F1 macro 0,980, 62 erreurs** sur 3 276, contre 71 pour la v3, 105 pour le modèle de la section 7 et 161 pour les forêts seules (−61 %).
- **Le gain sur les forêts est le plus fort sur les sessions courtes** : 0,766 → 0,851 → 0,894 (v3) → 0,872 (v5) sous 50 actions, 0,880 → 0,934 → **0,957** entre 51 et 100. C'est l'effet attendu de l'augmentation par extraits courts et de la règle du navigateur. Sous 50 actions, il n'y a que 47 sessions : la v5 y fait une erreur de plus que la v3, mais elle gagne sur toutes les tranches au-delà de 100 actions.
- Les confusions `skm` / `slq` / `flj` sont passées de 16 (forêts) à 12 (section 7), puis 9 (v3) et 10 (v5) : ces trois utilisateurs ont des profils très proches. Les autres confusions ne concernent plus que 1 ou 2 sessions chacune.
- Les utilisateurs les moins bien reconnus restent ceux qui ont **4 à 5 sessions** (`lra`, `jew`, `urs`, `bez`, `azn`) ou des profils proches d'un autre (`hll`, `azx`, confondus avec `guh`).

## 10. Interprétation

**Forêts** : l'importance d'une variable mesure de combien elle fait baisser l'impureté (le mélange des classes) en moyenne dans les nœuds où elle est utilisée ; on fait la moyenne sur les deux forêts.

**SVM** : chaque utilisateur a un vecteur de coefficients, un par mot ; les mots de plus fort coefficient sont ceux qui « votent » le plus pour cet utilisateur. On les regarde, pour le SVM avec temps, sur `skm` et `slq`, souvent confondus.

In [ ]:
forets.fit(X, y)
importances = pd.Series(np.mean([f.feature_importances_ for f in forets.estimators_], axis=0),
                        index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [2, 1]})
importances.head(20).sort_values().plot.barh(ax=ax[0], color='#2878b5')
ax[0].set_title('Forêts : les 20 variables les plus importantes')
ax[0].set_xlabel('Importance')

# Famille de chaque variable, calculée dans le même ordre que `importances` (qui est triée)
famille = importances.index.str.extract(r'^(nav|type|chaine|ecran|conf)_', expand=False).fillna('volume / temps / autres')
par_famille = importances.groupby(famille.values).sum().sort_values()
par_famille.plot.barh(ax=ax[1], color='#e07a5f')
ax[1].set_title('Importance totale par famille de variables')
ax[1].set_xlabel('Importance cumulée')
plt.tight_layout()
plt.show()
display((par_famille.sort_values(ascending=False) * 100).round(1).rename('% de l importance totale').to_frame())

In [ ]:
svm_temps = svm_tfidf(C=10).fit(documents_temps_train, y)
vocabulaire = svm_temps[0].get_feature_names_out()
coefficients = svm_temps[-1].coef_

for utilisateur in ['skm', 'slq']:
    k = list(svm_temps.classes_).index(utilisateur)
    meilleurs = np.argsort(coefficients[k])[::-1][:8]
    print(f'--- Mots qui votent le plus pour {utilisateur}')
    for j in meilleurs:
        print(f'  {coefficients[k, j]:5.2f}  {vocabulaire[j][:110]}')
print(f'Taille du vocabulaire TF-IDF : {len(vocabulaire):,} mots')
print('Mots contenant TEMPS parmi les 1 000 plus forts coefficients (tous utilisateurs) :',
      int(sum('TEMPS' in vocabulaire[j] for j in np.argsort(np.abs(coefficients).max(axis=0))[::-1][:1000])))

**À retenir :**
- **Forêts** : les **types d'actions** portent 38,9 % de l'importance avec seulement 35 colonnes (Raccourci, Retour sur un écran, Affichage d'un toast… les signatures de la partie 1), puis les écrans (24,9 %), les configurations (16,1 %) et les chaînes (10,7 %). Le **navigateur** ne pèse que 2,6 % alors qu'il sépare parfaitement les utilisateurs : l'importance par impureté favorise les variables continues et compte peu une variable binaire utilisée tôt et rarement (limite de cette mesure ; c'est pourquoi on impose la règle du navigateur explicitement).
- **SVM avec temps** : les mots qui votent le plus pour `skm` et `slq` sont des **paires d'actions précises** et des **configurations ou écrans propres** à chacun : pour `skm`, la configuration `ADM_055` et l'écran de saisie comptable `SaisiEcritForm` ; pour `slq`, la configuration `ChantierInstallation`, le visualiseur PDF, l'écran d'accueil.
- **Le temps compte** : des paires comme « TEMPS >> Raccourci(SaisiEcritForm) » (une action qui suit une pause) ou « Exécution d'un bouton(PdfViewerForm) >> TEMPS » (une action suivie d'une pause) figurent parmi les plus forts coefficients, et 170 des 1 000 mots les plus influents contiennent `TEMPS`. Le rythme de travail est bien une signature.

## 11. Prédiction sur le test et fichier de soumission

Les 7 SVM et les forêts sont entraînés sur **toutes** les sessions du train (les SVM augmentés sur les sessions et leurs extraits), appliqués aux 324 sessions du test, puis combinés comme la v5 : combinaison géométrique, correction des fréquences et règle du navigateur. Le fichier suit le format de `sample_submission.csv` : `RowId` (de 1 à 324, dans l'ordre du fichier de test) et `prediction` (le code de l'utilisateur).

*Durée : environ 7 minutes.*

In [ ]:
y_liste = list(y)
svms_finaux = [(svm_temps, documents_temps_test)]
for nom, graine, options in AUGMENTATIONS:
    documents_augmentes, y_augmente = augmenter(documents_temps_train, y_liste, np.random.default_rng(graine), **options)
    svms_finaux.append((svm_tfidf(C=10).fit(documents_augmentes, y_augmente), documents_temps_test))
    print(f'{nom} : {len(documents_augmentes)} documents ({len(y_liste)} sessions + leurs extraits)')
svms_finaux.append((svm_tfidf(C=3).fit(documents_train, y_liste), documents_test))
svms_finaux.append((svm_tfidf(C=10, use_idf=False).fit(documents_temps_train, y_liste), documents_temps_test))
for modele, _ in svms_finaux + [(forets, None)]:
    assert list(modele.classes_) == list(classes)

score_test = score_final([modele.decision_function(docs) for modele, docs in svms_finaux],
                         forets.predict_proba(X_test), masque_test)
preds_test = classes[score_test.argmax(axis=1)]
# Le score est une log-probabilité (à une constante près) : on le ramène à une probabilité pour lire la confiance
proba_test = softmax(score_test, 1.0)
confiance = proba_test.max(axis=1)

soumission = pd.DataFrame({'RowId': X_test.index + 1, 'prediction': preds_test})
soumission.to_csv(DATA_DIR / 'submission.csv', index=False)

modele_sample = pd.read_csv(trouver_fichier(DATA_DIR, 'sample_submission'))
assert list(soumission.columns) == list(modele_sample.columns)
assert soumission['RowId'].tolist() == modele_sample['RowId'].tolist()
print(f"submission.csv écrit : {len(soumission)} lignes, {soumission['prediction'].nunique()} utilisateurs différents prédits")

# Contrôle : identique à la soumission notée 0,991 sur Kaggle ? Et combien de différences avec la v3 et la v4 ?
for nom_fichier, score_kaggle in [('submission_v5_geometrique.csv', '0,991'), ('submission_v4_7svm.csv', '0,985'),
                                  ('submission_v3_3svm.csv', '0,985')]:
    for chemin in [DATA_DIR / 'soumissions' / nom_fichier, DATA_DIR / nom_fichier]:
        if chemin.exists():
            reference = pd.read_csv(chemin)
            print(f'Différences avec {nom_fichier} ({score_kaggle} sur Kaggle) :',
                  int((reference['prediction'] != soumission['prediction']).sum()))
            break
display(soumission.head())

In [ ]:
nav_test = X_test[nav_cols].idxmax(axis=1)
coherent = (nav_test.values == navigateur_par_utilisateur.loc[preds_test].values).mean()
print(f'Sessions du test dont l utilisateur prédit a le même navigateur : {coherent:.1%}')

plt.figure(figsize=(8, 4))
sns.histplot(confiance, bins=20, color='#4d908e')
plt.title('Confiance du modèle final sur les sessions du test')
plt.xlabel('Probabilité combinée attribuée à l utilisateur prédit')
plt.ylabel('Nombre de sessions')
plt.tight_layout()
plt.show()
print(f'Confiance médiane : {np.median(confiance):.2f} ; sessions avec une confiance < 0,2 : {(confiance < 0.2).sum()}')

**À retenir :**
- `submission.csv` contient les **324 prédictions** au format attendu ; 227 utilisateurs différents sont prédits. Il est **identique** à la soumission notée **0,991** sur Kaggle : le notebook reproduit exactement le meilleur résultat. Il ne diffère de la v3 et de la v4 (0,985) que de 3 prédictions.
- **100 %** des sessions du test sont attribuées à un utilisateur qui a le même navigateur (règle imposée).
- La confiance médiane (0,13) est basse : la moyenne géométrique de pseudo-probabilités déjà « douces » (T = 1) est très aplatie. Ce n'est pas une perte de qualité : ce qui compte est l'ordre des utilisateurs, pas la valeur absolue du score.